# 03. HELOC Counterfactual Analysis

HELOC 用に学習したロジスティック回帰モデルを DiCE に渡し、リスクの高い事例に対して反事実を生成します。  
介入特徴は HELOC の特徴量体系に合わせて設定し、現実的な変化方向だけを許可します。


## セットアップ


In [14]:
%pip install pandas numpy scikit-learn joblib dice-ml pyarrow plotly xgboost shap


Note: you may need to restart the kernel to use updated packages.


In [15]:
import dice_ml
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", 200)


## データと artifact のロード


In [16]:
BASE_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "blog" / "01.Counterfactual_Analysis",
    Path("/home/kohei/WorkSpace/blog/01.Counterfactual_Analysis"),
]

BASE_DIR = next(
    (path for path in BASE_DIR_CANDIDATES if (path / "model" / "heloc_logistic_artifact.joblib").exists()),
    None,
)
if BASE_DIR is None:
    raise FileNotFoundError("heloc_logistic_artifact.joblib が見つかりません。02.heloc_prediction_model.ipynb を先に実行してください。")

DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "model"

TRAIN_PATH = DATA_DIR / "train_df.parquet"
TEST_PATH = DATA_DIR / "test_df.parquet"
ARTIFACT_PATH = MODEL_DIR / "heloc_logistic_artifact.joblib"
FEATURE_METADATA_PATH = DATA_DIR / "heloc_feature_metadata.csv"

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)
artifact = joblib.load(ARTIFACT_PATH)
feature_metadata_df = pd.read_csv(FEATURE_METADATA_PATH)

model = artifact["model"]
feature_cols = artifact["feature_cols"]
target_col = artifact["target_col"]
class_names = artifact["class_names"]
class_mapping = artifact["class_mapping"]
best_threshold = artifact["best_threshold"]

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("feature count:", len(feature_cols))
print("target column:", target_col)
print("class mapping:", class_mapping)
print(f"best threshold: {best_threshold:.4f}")


train shape: (7888, 47)
test shape: (1983, 47)
feature count: 46
target column: is_at_risk
class mapping: {np.int64(0): np.int64(0), np.int64(1): np.int64(1)}
best threshold: 0.3700


In [17]:
missing_in_train = sorted(set(feature_cols + [target_col]) - set(train_df.columns))
missing_in_test = sorted(set(feature_cols + [target_col]) - set(test_df.columns))

if missing_in_train or missing_in_test:
    raise ValueError(f"Missing columns. train={missing_in_train}, test={missing_in_test}")

model_train_df = train_df[feature_cols + [target_col]].copy()
model_test_df = test_df[feature_cols + [target_col]].copy()

display(model_train_df.head())
display(feature_metadata_df.head(20))


,average_duration_of_resolution,balance_trade_ratio_log1p,credit_history_length,delinquency_trade_ratio_log1p,delinquency_trade_sum_log1p,estimate_of_risk,estimate_of_risk_is_special_9,high_ratio_bank_share_log1p,illegal_trade_gap,inquiry_pressure_log1p,maximum_illegal_trades,maximum_illegal_trades_over_last_year,months_since_first_trade,months_since_first_trade_is_special_8,months_since_last_illegal_trade_is_special_7,months_since_last_illegal_trade_is_special_8,months_since_last_illegal_trade_log1p,months_since_last_inquiry_not_recent_is_special_7,months_since_last_inquiry_not_recent_is_special_8,months_since_last_inquiry_not_recent_log1p,months_since_last_trade_log1p,net_fraction_of_installment_burden,net_fraction_of_installment_burden_is_special_8,net_fraction_of_revolving_burden,net_fraction_of_revolving_burden_is_special_8,nr_banks_with_high_ratio_is_special_8,nr_banks_with_high_ratio_log1p,nr_inquiries_in_last_6_months_log1p,nr_inquiries_in_last_6_months_not_recent_log1p,nr_installment_trades_with_balance_is_special_8,nr_installment_trades_with_balance_log1p,nr_revolving_trades_with_balance_is_special_8,nr_revolving_trades_with_balance_log1p,nr_total_trades,nr_trades_initiated_in_last_year_log1p,nr_trades_insolvent_for_over_60_days_log1p,nr_trades_insolvent_for_over_90_days_log1p,number_of_satisfactory_trades,percentage_of_installment_trades,percentage_of_legal_trades,percentage_trades_with_balance,percentage_trades_with_balance_is_special_8,recent_activity_gap,recent_inquiry_share,revolving_installment_burden_gap,satisfactory_trade_ratio_log1p,is_at_risk
0,63.0,0.374693,136.0,0.0,0.0,80.0,0,0.000000,0.0,0.087011,6.0,6.0,136.0,0,0,0,3.091042,1,0,0.000000,2.772589,54.0,0,19.0,0,0,0.000000,0.693147,0.693147,0,1.386294,0,1.098612,11.0,0.0,0.0,0.0,11.0,36.0,91.0,71.0,0,121.0,1.0,-35.0,0.693147,0
1,130.0,0.251314,342.0,0.0,0.0,91.0,0,0.000000,1.0,0.000000,8.0,7.0,342.0,0,1,0,0.000000,0,1,0.000000,3.761200,74.0,1,3.0,0,0,0.000000,0.000000,0.000000,0,1.098612,0,1.098612,14.0,0.0,0.0,0.0,14.0,36.0,100.0,50.0,0,300.0,0.0,-71.0,0.693147,0
2,88.0,0.361013,193.0,0.0,0.0,56.0,0,0.160343,2.0,0.042560,6.0,4.0,193.0,0,0,0,1.386294,0,0,0.000000,3.850148,37.0,0,64.0,0,0,1.609438,0.693147,0.693147,0,1.098612,0,2.197225,23.0,0.0,0.0,0.0,22.0,22.0,78.0,83.0,0,147.0,1.0,27.0,0.671168,1
3,76.0,0.356675,250.0,0.0,0.0,63.0,0,0.000000,2.0,0.133531,6.0,4.0,250.0,0,0,0,0.693147,0,0,0.000000,3.332205,11.0,0,36.0,0,0,0.000000,1.098612,1.098612,0,0.693147,0,1.791759,14.0,0.0,0.0,0.0,12.0,17.0,75.0,75.0,0,223.0,1.0,25.0,0.619039,1
4,60.0,0.295464,255.0,0.0,0.0,72.0,0,0.247836,1.0,0.000000,8.0,7.0,255.0,0,1,0,0.000000,0,0,2.639057,2.639057,14.0,0,85.0,0,0,2.302585,0.000000,0.000000,0,1.098612,0,2.302585,32.0,0.0,0.0,0.0,31.0,22.0,100.0,79.0,0,242.0,0.0,71.0,0.677399,1


,feature,role,dtype,description,in_model_features
0,average_duration_of_resolution,base_numeric,float64,口座や取引の平均継続期間を表す指標です。,True
1,balance_trade_ratio_log1p,log_numeric,float64,balance_trade_ratio に対して log1p 変換を適用した列。,True
2,credit_history_length,derived_numeric,float64,初回取引からの経過月数。,True
3,delinquency_trade_ratio_log1p,log_numeric,float64,delinquency_trade_ratio に対して log1p 変換を適用した列。,True
4,delinquency_trade_sum_log1p,log_numeric,float64,delinquency_trade_sum に対して log1p 変換を適用した列。,True
5,estimate_of_risk,base_numeric,float64,申込者の信用リスクの推定スコア。大きいほど低リスク寄りと解釈されることが多いです。,True
6,estimate_of_risk_is_special_9,special_value_flag,int64,estimate_of_risk が特殊値 -9 を持っていたことを表すフラグ。意味: 信用...,True
7,high_ratio_bank_share_log1p,log_numeric,float64,high_ratio_bank_share に対して log1p 変換を適用した列。,True
8,illegal_trade_gap,derived_numeric,float64,全期間の問題取引最大件数と過去1年の最大件数の差。,True
9,inquiry_pressure_log1p,log_numeric,float64,inquiry_pressure に対して log1p 変換を適用した列。,True


## DiCE に渡すロジスティック回帰ラッパー


In [18]:
class DiceReadyModel:
    def __init__(self, model, feature_names, threshold):
        self.model = model
        self.feature_names = feature_names
        self.threshold = threshold
        self.classes_ = np.array([0, 1])

    def _prepare(self, input_df: pd.DataFrame) -> pd.DataFrame:
        return input_df[self.feature_names].copy()

    def predict_proba(self, input_df: pd.DataFrame) -> np.ndarray:
        X = self._prepare(input_df)
        return self.model.predict_proba(X)

    def predict(self, input_df: pd.DataFrame) -> np.ndarray:
        positive_proba = self.predict_proba(input_df)[:, 1]
        return (positive_proba >= self.threshold).astype(int)


dice_ready_model = DiceReadyModel(
    model=model,
    feature_names=feature_cols,
    threshold=best_threshold,
)

dice_data = dice_ml.Data(
    dataframe=model_train_df,
    continuous_features=feature_cols,
    outcome_name=target_col,
)

dice_model = dice_ml.Model(
    model=dice_ready_model,
    backend="sklearn",
    model_type="classifier",
)

dice = dice_ml.Dice(dice_data, dice_model, method="genetic")


## 介入特徴の設定


In [19]:
# HELOC では、残高負担や照会圧力の軽減に相当する列だけを介入対象にする。
# いずれも学習に使われた最終特徴量空間の列名を指定する。

actionable_features = [
    "net_fraction_of_revolving_burden",
    "net_fraction_of_installment_burden",
    "percentage_trades_with_balance",
    "nr_inquiries_in_last_6_months_log1p",
    "high_ratio_bank_share_log1p",
]

monotone_decrease_features = [
    "percentage_trades_with_balance",
    "nr_inquiries_in_last_6_months_log1p",
    "high_ratio_bank_share_log1p",
]
monotone_increase_features = []

domain_bounds = {
    "net_fraction_of_revolving_burden": (0.0, None),
    "net_fraction_of_installment_burden": (0.0, None),
    "percentage_trades_with_balance": (0.0, 100.0),
    "nr_inquiries_in_last_6_months_log1p": (0.0, None),
    "high_ratio_bank_share_log1p": (0.0, None),
}

display_name_map = {
    "net_fraction_of_revolving_burden": "net_fraction_of_revolving_burden",
    "net_fraction_of_installment_burden": "net_fraction_of_installment_burden",
    "percentage_trades_with_balance": "percentage_trades_with_balance",
    "nr_inquiries_in_last_6_months_log1p": "nr_inquiries_in_last_6_months",
    "high_ratio_bank_share_log1p": "high_ratio_bank_share",
}

actionable_feature_metadata_df = feature_metadata_df.loc[
    feature_metadata_df["feature"].isin(actionable_features),
    ["feature", "role", "description"],
].copy()

print("actionable feature count:", len(actionable_features))
display(actionable_feature_metadata_df)


actionable feature count: 5


,feature,role,description
7,high_ratio_bank_share_log1p,log_numeric,high_ratio_bank_share に対して log1p 変換を適用した列。
21,net_fraction_of_installment_burden,base_numeric,分割払い型口座の負担度合いを表す比率的な指標です。
23,net_fraction_of_revolving_burden,base_numeric,リボルビング型口座の負担度合いを表す比率的な指標です。
27,nr_inquiries_in_last_6_months_log1p,log_numeric,nr_inquiries_in_last_6_months に対して log1p 変換を適用...
40,percentage_trades_with_balance,base_numeric,残高がある取引の割合。


In [20]:
def build_permitted_range(df: pd.DataFrame, query_df: pd.DataFrame, columns: list[str]) -> dict:
    permitted = {}
    query_row = query_df.iloc[0]

    for col in columns:
        lower = float(df[col].min())
        upper = float(df[col].max())

        if col in domain_bounds:
            domain_lower, domain_upper = domain_bounds[col]
            if domain_lower is not None:
                lower = max(lower, float(domain_lower))
            if domain_upper is not None:
                upper = min(upper, float(domain_upper))

        if col in monotone_decrease_features:
            upper = min(upper, float(query_row[col]))

        if col in monotone_increase_features:
            lower = max(lower, float(query_row[col]))

        if upper < lower:
            upper = lower

        permitted[col] = [lower, upper]

    return permitted


## テストデータのスコアリング


In [21]:
scored_test_df = model_test_df.copy()
scored_test_df["risk_proba"] = dice_ready_model.predict_proba(scored_test_df[feature_cols])[:, 1]
scored_test_df["predicted_risk"] = dice_ready_model.predict(scored_test_df[feature_cols])

display(scored_test_df[[target_col, "risk_proba", "predicted_risk"]].head())
print("share predicted as risky in test:", scored_test_df["predicted_risk"].mean())


,is_at_risk,risk_proba,predicted_risk
0,1,4.660017e-06,0
1,0,3.440198e-17,0
2,0,5.877229e-29,0
3,0,3.017181e-34,0
4,1,5.801077e-14,0


share predicted as risky in test: 0.01059001512859304


## 分析対象ケースの選定


In [22]:
# 実運用の否決候補に近づけるため、実測ラベルではなく陽性判定全体から代表ケースを選ぶ。
positive_candidate_df = scored_test_df.loc[
    scored_test_df["predicted_risk"] == 1
].copy()
positive_candidate_df["distance_to_threshold"] = positive_candidate_df["risk_proba"] - best_threshold


def pick_case_index(candidate_df: pd.DataFrame, target_proba: float | None = None, mode: str = "nearest") -> int:
    if candidate_df.empty:
        raise ValueError("No positive candidates were found in test data.")
    if mode == "highest":
        return int(candidate_df["risk_proba"].idxmax())
    if target_proba is None:
        raise ValueError("target_proba is required unless mode='highest'")
    return int((candidate_df["risk_proba"] - target_proba).abs().idxmin())


case_definitions = [
    {"case_id": "high", "label": "High", "mode": "nearest", "target_proba": 0.85},
    {"case_id": "mid", "label": "Mid", "mode": "nearest", "target_proba": 0.70},
    {"case_id": "low", "label": "Low", "mode": "nearest", "target_proba": 0.55},
]

case_rows = []
for case in case_definitions:
    case_index = pick_case_index(
        positive_candidate_df,
        target_proba=case["target_proba"],
        mode=case["mode"],
    )
    case_rows.append(
        {
            "case_id": case["case_id"],
            "case_label": case["label"],
            "index": case_index,
            "risk_proba": float(positive_candidate_df.loc[case_index, "risk_proba"]),
            "distance_to_threshold": float(positive_candidate_df.loc[case_index, "distance_to_threshold"]),
        }
    )

case_summary_df = pd.DataFrame(case_rows)
display(case_summary_df)

selected_case_preview_df = (
    case_summary_df.set_index("case_label")[["index", "risk_proba", "distance_to_threshold"]]
    .join(model_test_df.loc[case_summary_df["index"], actionable_features].set_index(case_summary_df["case_label"]))
)
display(selected_case_preview_df)


,case_id,case_label,index,risk_proba,distance_to_threshold
0,high,High,609,0.869991,0.499991
1,mid,Mid,60,0.621734,0.251734
2,low,Low,235,0.598886,0.228886


,index,risk_proba,distance_to_threshold,net_fraction_of_revolving_burden,net_fraction_of_installment_burden,percentage_trades_with_balance,nr_inquiries_in_last_6_months_log1p,high_ratio_bank_share_log1p
case_label,,,,,,,,
High,609,0.869991,0.499991,99.0,38.0,100.0,0.000000,0.167054
Mid,60,0.621734,0.251734,30.0,74.0,100.0,0.000000,0.117783
Low,235,0.598886,0.228886,76.0,83.0,100.0,0.693147,0.095310


## 反事実生成のヘルパー


In [23]:
desired_risk_class = 0

# DiCE の到達判定をモデル本体の分類閾値に合わせる。
case_generation_config = {
    "name": "heloc_logistic_baseline_3cf",
    "features": actionable_features,
    "total_CFs": 3,
    "diversity_weight": 20.0,
    "proximity_weight": 0.5,
    "stopping_threshold": best_threshold,
}


def inverse_display_value(feature_name: str, value: float) -> float:
    if feature_name.endswith("_log1p"):
        return float(np.expm1(value))
    return float(value)


def summarize_changes(query_df: pd.DataFrame, cf_df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    base_row = query_df.iloc[0]
    records = []

    for cf_idx, (_, cf_row) in enumerate(cf_df.iterrows(), start=1):
        for col in columns:
            before = inverse_display_value(col, float(base_row[col]))
            after = inverse_display_value(col, float(cf_row[col]))
            if np.isclose(before, after):
                continue
            records.append(
                {
                    "cf_id": f"シナリオ{cf_idx}",
                    "feature": display_name_map.get(col, col),
                    "model_feature": col,
                    "before": before,
                    "after": after,
                    "delta": after - before,
                    "abs_delta": abs(after - before),
                }
            )

    return pd.DataFrame(records)


def build_scenario_value_matrix(
    query_df: pd.DataFrame,
    cf_df: pd.DataFrame,
    columns: list[str],
) -> pd.DataFrame:
    scenario_labels = ["original"] + [f"シナリオ{i}" for i in range(1, len(cf_df) + 1)]
    value_matrix = pd.concat([query_df[columns], cf_df[columns]], axis=0).copy()
    value_matrix = value_matrix.apply(
        lambda col: col.map(lambda value: inverse_display_value(col.name, float(value)))
    )
    value_matrix = value_matrix.rename(columns=display_name_map)
    value_matrix.index = scenario_labels
    return value_matrix


def style_value_matrix(value_matrix: pd.DataFrame, caption: str = ""):
    delta_matrix = value_matrix.subtract(value_matrix.loc["original"], axis=1)
    max_abs = float(np.abs(delta_matrix.to_numpy()).max())
    if max_abs == 0:
        max_abs = 1.0

    styler = value_matrix.style.format("{:.4f}").background_gradient(
        cmap="RdBu_r",
        axis=None,
        gmap=delta_matrix,
        vmin=-max_abs,
        vmax=max_abs,
    )
    if caption:
        styler = styler.set_caption(caption)
    return styler


def plot_scenario_value_dotplot(value_matrix: pd.DataFrame, title: str):
    delta_df = value_matrix.subtract(value_matrix.loc["original"], axis=1).T.iloc[::-1]
    axis_limit = float(np.abs(delta_df.to_numpy()).max())
    if axis_limit == 0:
        axis_limit = 1.0

    palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
    symbols = ["circle", "square", "diamond", "triangle-up", "cross", "x"]

    fig = go.Figure()
    for idx, scenario in enumerate(delta_df.columns):
        fig.add_trace(
            go.Scatter(
                x=delta_df[scenario].values,
                y=delta_df.index.tolist(),
                mode="markers",
                name=scenario,
                marker=dict(size=12, color=palette[idx % len(palette)], symbol=symbols[idx % len(symbols)]),
                hovertemplate="feature=%{y}<br>delta=%{x:.4f}<extra>" + scenario + "</extra>",
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title="Delta from original",
        yaxis_title="Feature",
        width=1100,
        height=max(350, 55 * len(delta_df.index)),
        xaxis=dict(range=[-axis_limit * 1.1, axis_limit * 1.1]),
        template="plotly_white",
    )
    fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="black")
    fig.show()


def generate_case_counterfactuals(case_index: int):
    case_query = model_test_df.loc[[case_index], feature_cols].copy()
    case_features = case_generation_config["features"]
    case_permitted_range = build_permitted_range(model_train_df, case_query, case_features)

    case_counterfactuals = dice.generate_counterfactuals(
        query_instances=case_query,
        total_CFs=case_generation_config["total_CFs"],
        desired_class=desired_risk_class,
        features_to_vary=case_features,
        permitted_range=case_permitted_range,
        stopping_threshold=case_generation_config["stopping_threshold"],
        diversity_weight=case_generation_config["diversity_weight"],
        proximity_weight=case_generation_config["proximity_weight"],
    )

    case_cf_df = case_counterfactuals.cf_examples_list[0].final_cfs_df.copy()
    case_change_summary = summarize_changes(case_query, case_cf_df, case_features)
    case_value_matrix = build_scenario_value_matrix(case_query, case_cf_df, case_features)

    return case_query, case_cf_df, case_change_summary, case_value_matrix


## 反事実生成の実行


In [24]:
case_result_frames = []
case_status_rows = []
case_value_matrices = {}

for row in case_rows:
    try:
        case_query, case_cf_df, case_change_summary, case_value_matrix = generate_case_counterfactuals(row["index"])
    except Exception as exc:
        case_status_rows.append(
            {
                "case_id": row["case_id"],
                "case_label": row["case_label"],
                "index": row["index"],
                "risk_proba": row["risk_proba"],
                "status": "failed",
                "detail": str(exc),
            }
        )
        continue

    case_status_rows.append(
        {
            "case_id": row["case_id"],
            "case_label": row["case_label"],
            "index": row["index"],
            "risk_proba": row["risk_proba"],
            "status": "ok",
            "detail": "",
        }
    )

    case_result = case_change_summary.copy()
    case_result.insert(0, "case_id", row["case_id"])
    case_result.insert(1, "case_label", row["case_label"])
    case_result.insert(2, "query_index", row["index"])
    case_result.insert(3, "original_risk_proba", row["risk_proba"])
    case_result_frames.append(case_result)
    case_value_matrices[row["case_label"]] = case_value_matrix

case_status_df = pd.DataFrame(case_status_rows)
case_comparison_df = pd.concat(case_result_frames, axis=0, ignore_index=True) if case_result_frames else pd.DataFrame()

display(case_status_df)
display(case_comparison_df)


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  5.73it/s]


,case_id,case_label,index,risk_proba,status,detail
0,high,High,609,0.869991,ok,
1,mid,Mid,60,0.621734,ok,
2,low,Low,235,0.598886,ok,


,case_id,case_label,query_index,original_risk_proba,cf_id,feature,model_feature,before,after,delta,abs_delta
0,high,High,609,0.869991,シナリオ1,net_fraction_of_revolving_burden,net_fraction_of_revolving_burden,99.000000,82.000000,-17.000000,17.000000
1,high,High,609,0.869991,シナリオ1,net_fraction_of_installment_burden,net_fraction_of_installment_burden,38.000000,42.000000,4.000000,4.000000
2,high,High,609,0.869991,シナリオ1,high_ratio_bank_share,high_ratio_bank_share_log1p,0.181818,0.105171,-0.076647,0.076647
3,high,High,609,0.869991,シナリオ2,net_fraction_of_revolving_burden,net_fraction_of_revolving_burden,99.000000,83.000000,-16.000000,16.000000
4,high,High,609,0.869991,シナリオ2,net_fraction_of_installment_burden,net_fraction_of_installment_burden,38.000000,42.000000,4.000000,4.000000
5,high,High,609,0.869991,シナリオ2,high_ratio_bank_share,high_ratio_bank_share_log1p,0.181818,0.105171,-0.076647,0.076647
6,high,High,609,0.869991,シナリオ3,net_fraction_of_revolving_burden,net_fraction_of_revolving_burden,99.000000,91.000000,-8.000000,8.000000
7,high,High,609,0.869991,シナリオ3,net_fraction_of_installment_burden,net_fraction_of_installment_burden,38.000000,17.000000,-21.000000,21.000000
8,high,High,609,0.869991,シナリオ3,high_ratio_bank_share,high_ratio_bank_share_log1p,0.181818,0.105171,-0.076647,0.076647
9,mid,Mid,60,0.621734,シナリオ1,net_fraction_of_revolving_burden,net_fraction_of_revolving_burden,30.000000,17.200000,-12.800000,12.800000


## 反事実の表示


In [25]:
if not case_comparison_df.empty:
    case_delta_matrix = (
        case_comparison_df.pivot_table(
            index=["case_label", "cf_id"],
            columns="feature",
            values="delta",
            aggfunc="first",
        )
        .reindex(columns=actionable_features)
        .fillna(0)
    )

    display(
        case_delta_matrix.style
        .format("{:.4f}")
        .background_gradient(
            cmap="RdBu_r",
            axis=None,
            vmin=-max(1.0, float(np.abs(case_delta_matrix.to_numpy()).max())),
            vmax=max(1.0, float(np.abs(case_delta_matrix.to_numpy()).max())),
        )
    )


In [26]:
for case_label, value_matrix in case_value_matrices.items():
    display(
        style_value_matrix(
            value_matrix,
            caption=f"{case_label} case: actionable feature values",
        )
    )
    plot_scenario_value_dotplot(
        value_matrix,
        title=f"{case_label} Case Scenarios (x: delta from original)",
    )


,net_fraction_of_revolving_burden,net_fraction_of_installment_burden,percentage_trades_with_balance,nr_inquiries_in_last_6_months,high_ratio_bank_share
original,99.0000,38.0000,100.0000,0.0000,0.1818
シナリオ1,82.0000,42.0000,100.0000,0.0000,0.1052
シナリオ2,83.0000,42.0000,100.0000,0.0000,0.1052
シナリオ3,91.0000,17.0000,100.0000,0.0000,0.1052


,net_fraction_of_revolving_burden,net_fraction_of_installment_burden,percentage_trades_with_balance,nr_inquiries_in_last_6_months,high_ratio_bank_share
original,30.0000,74.0000,100.0000,0.0000,0.1250
シナリオ1,17.2000,45.9000,91.7000,0.0000,0.1052
シナリオ2,20.2000,32.8000,61.0000,0.0000,0.1052
シナリオ3,12.0000,74.0000,67.0000,0.0000,0.0000


,net_fraction_of_revolving_burden,net_fraction_of_installment_burden,percentage_trades_with_balance,nr_inquiries_in_last_6_months,high_ratio_bank_share
original,76.0000,83.0000,100.0000,1.0000,0.1000
シナリオ1,76.0000,74.0000,100.0000,0.4918,0.1052
シナリオ2,30.0000,58.0000,100.0000,1.0138,0.1052
シナリオ3,58.0000,77.0000,86.0000,1.0138,0.1052
